Neden tek hava sütunu

Bu modelde basitleştirmenin sürekli kazandırdığını görmüştük
(yaprak 63 -> 31 -> 15 hep iyileştirdi), dolayısıyla 15 sütun eklemek o yönle çelişiyordu.

Üç ayrı doğrulamada tek tek ölçünce en değerli sütunun 7 günlük ortalama sıcaklık
olduğu çıktı, ve test rejimine en yakın doğrulamada tek sütun 15 sütunluk paketten
daha iyi sonuç verdi (0.7600 vs 0.7619). Kaggle bunu doğruladı.

sic_7gun'ü günlük sıcaklığa tercih etmemizin sebebi termal atalet: binalar bir günde
ısınıp soğumaz, üç gündür sıcaksa klimalar sürekli çalışıyordur. Ham günlük sıcaklık
her doğrulamada 7 günlük ortalamanın gerisinde kaldı.

Ayrıca CDD (soğutma derece-gün) yerine ham sıcaklığı seçtik: test dönemi Nisan-Temmuz ve
Nisan-Mayıs'ta CDD sıfıra yakın, yani o aylarda hiç bilgi taşımıyor; ham sıcaklık ise
orada da değişiyor.

In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb

RAW, EXT = '../data/raw/', '../data/external/'
train = pd.read_csv(RAW + 'train.csv')
test  = pd.read_csv(RAW + 'test.csv')
train['tarih'] = pd.to_datetime(train['tarih'])
test['tarih']  = pd.to_datetime(test['tarih'])

# EDA'da bulduğumuz 444 satır, kurulu gücün 24 saatlik teorik tavanını aşıyor (2-58 kat).
# %78'i sadece 4 trafoda toplandığı için global kırpma yerine trafo bazlı medyanla düzeltiyoruz.
train['asiri_oran'] = train['tuketim'] / (train['guc'] * 24)
train['tuketim_duzeltilmis'] = train['tuketim']
for t in train.loc[train['asiri_oran'] > 2, 'tanim'].unique():
    medyan = train.loc[(train['tanim'] == t) & (train['asiri_oran'] <= 2), 'tuketim'].median()
    train.loc[(train['tanim'] == t) & (train['asiri_oran'] > 2), 'tuketim_duzeltilmis'] = medyan

print('düzeltilen satır:', (train['tuketim'] != train['tuketim_duzeltilmis']).sum())

düzeltilen satır: 444


In [2]:
TATILLER = pd.to_datetime([
    '2025-01-01','2025-03-29','2025-03-30','2025-03-31','2025-04-01','2025-04-23','2025-05-01',
    '2025-05-19','2025-06-05','2025-06-06','2025-06-07','2025-06-08','2025-06-09','2025-08-30','2025-10-29',
    '2026-01-01','2026-03-19','2026-03-20','2026-03-21','2026-04-23','2026-05-01','2026-05-19',
    '2026-05-26','2026-05-27','2026-05-28','2026-05-29','2026-05-30','2026-08-30','2026-10-29',
]).date

GUC_BINS = [0, 100, 250, 500, 1000, 2000, 5000, np.inf]


def temel_ozellikler(df):
    df = df.copy()
    lok = df['lokasyon'].str.split('>', expand=True)
    df['il']    = lok[0]
    df['bolge'] = lok[1]
    # Manisa'da ilçe kırılımı yok; orada bölge adı hem kategori hem hava anahtarı oluyor
    df['ilce']  = (lok[2] if lok.shape[1] > 2 else pd.Series(np.nan, index=df.index)).fillna('YOK')
    df['ay']            = df['tarih'].dt.month
    df['gun']           = df['tarih'].dt.day
    df['haftanin_gunu'] = df['tarih'].dt.dayofweek
    df['hafta_sonu']    = (df['haftanin_gunu'] >= 5).astype(int)
    df['tatil']         = df['tarih'].dt.date.isin(TATILLER).astype(int)
    df['guc_bucket']     = pd.cut(df['guc'], bins=GUC_BINS)
    df['guc_bucket_str'] = df['guc_bucket'].astype(str)
    df['yer'] = np.where(df['ilce'] != 'YOK', df['ilce'], df['bolge'])
    return df


train = temel_ozellikler(train)
test  = temel_ozellikler(test)
train['log_t'] = np.log1p(train['tuketim_duzeltilmis'])
KATEGORIK = ['il', 'bolge', 'ilce', 'guc_bucket_str']

# yilin_gunu BİLEREK yok: eğitim ayları (4-9, 12-3) ile test ayları (4-7) arasında
# değer aralığı örtüşmüyor, ağaçlar extrapolasyon yapamıyor. Çıkarınca skor düzelmişti.
print('train', train.shape, '| test', test.shape)

train (1226237, 19) | test (714688, 16)


In [3]:
# Hava durumu: Open-Meteo arşiv API'si, 47 ilçe/bölge, Ocak 2025 - Temmuz 2026.
# Test dönemi geçmişte kaldığı için tahmin değil GERÇEKLEŞMİŞ sıcaklıkları kullanıyoruz.
# Çekim kodu 03_deneyler.ipynb'de; burada hazır dosyayı okuyoruz.
hava = pd.read_csv(EXT + 'hava_durumu.csv', parse_dates=['tarih'])
hava = hava.sort_values(['yer', 'tarih']).reset_index(drop=True)

# Termal atalet: 7 günlük hareketli ortalama. Her lokasyon kendi içinde hesaplanmalı.
hava['sic_7gun'] = hava.groupby('yer')['sic_ort'].transform(
    lambda s: s.rolling(7, min_periods=1).mean())

train = train.merge(hava[['yer', 'tarih', 'sic_7gun']], on=['yer', 'tarih'], how='left')
test  = test.merge(hava[['yer', 'tarih', 'sic_7gun']],  on=['yer', 'tarih'], how='left')

print('eksik sıcaklık - train:', train['sic_7gun'].isna().sum(), '| test:', test['sic_7gun'].isna().sum())
print()
# yilin_gunu'nun aksine sıcaklık aralıkları örtüşüyor; ağaçlar extrapolasyon yapmak zorunda değil
print('sic_7gun  train: %.1f – %.1f' % (train['sic_7gun'].min(), train['sic_7gun'].max()))
print('          test : %.1f – %.1f' % (test['sic_7gun'].min(), test['sic_7gun'].max()))

eksik sıcaklık - train: 0 | test: 0

sic_7gun  train: -2.9 – 34.7
          test : 5.7 – 31.3


Özellikler

Hepsi origin tarihinden önceki veriden hesaplanıyor(sızıntı karşıtı koruma)

- Grup istatistiği: (il, bölge, güç kovası) bazında log-ortalama, shrinkage'lı.
  Küçük gruplarda ham ortalama gürültülü (97 gruptan 3'ü 50 satırın altında, birinin
  ham ortalaması sıfır çıkmıştı), k=30 ile genel ortalamaya doğru çekiyoruz.
  Eşleşmeyen kombinasyonlar için il+güç, sonra genel ortalama yedeği var.
- Trafo profili: ortalama, medyan, gün sayısı, log ortalama/medyan/std, sıfır oranı,
  son 30 ve son 90 günün log ortalaması ve sıfır oranı, kısa/uzun vade farkı (log_trend).
- Hafta günü profili: trafonun kendi ortalamasına göre gün bazlı oranı.
  Sanayi trafosu ile konut trafosunu ayırıyor.

In [4]:
def trafo_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin]
    profil = g.groupby('tanim').agg(
        trafo_ortalama   = ('tuketim_duzeltilmis', 'mean'),
        trafo_medyan     = ('tuketim_duzeltilmis', 'median'),
        trafo_gun_sayisi = ('tuketim_duzeltilmis', 'count'),
        log_ortalama     = ('log_t', 'mean'),
        log_medyan       = ('log_t', 'median'),
        log_std          = ('log_t', 'std'),
        sifir_orani      = ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
    ).reset_index()
    for gun in [30, 90]:
        son = g[g['tarih'] >= origin - pd.Timedelta(days=gun)]
        ozet = son.groupby('tanim').agg(**{
            f'son{gun}_log_ort': ('log_t', 'mean'),
            f'son{gun}_sifir'  : ('tuketim_duzeltilmis', lambda x: (x == 0).mean()),
        }).reset_index()
        profil = profil.merge(ozet, on='tanim', how='left')
    profil['log_trend'] = profil['son30_log_ort'] - profil['son90_log_ort']
    return profil


def trafo_gun_profili(gecmis, origin):
    g = gecmis[gecmis['tarih'] < origin][['tanim', 'haftanin_gunu', 'tuketim_duzeltilmis']].copy()
    ort = g.groupby('tanim')['tuketim_duzeltilmis'].mean().rename('_ort')
    g = g.merge(ort, on='tanim', how='left')
    g['oran'] = g['tuketim_duzeltilmis'] / g['_ort'].replace(0, np.nan)
    return g.groupby(['tanim', 'haftanin_gunu'])['oran'].mean().rename('trafo_gun_orani').reset_index()


def veri_seti_kur(hedef, origin, k=30):
    g = train[train['tarih'] < origin]
    genel = g['log_t'].mean()

    grup = g.groupby(['il', 'bolge', 'guc_bucket'], observed=True)['log_t'].agg(['sum', 'count']).reset_index()
    grup['grup_log_ort'] = (grup['sum'] + k * genel) / (grup['count'] + k)
    il_guc = g.groupby(['il', 'guc_bucket'], observed=True)['log_t'].mean().rename('ilguc_log_ort').reset_index()

    df = hedef.merge(grup[['il', 'bolge', 'guc_bucket', 'grup_log_ort']],
                     on=['il', 'bolge', 'guc_bucket'], how='left')
    df = df.merge(il_guc, on=['il', 'guc_bucket'], how='left')
    df['grup_log_ort'] = df['grup_log_ort'].fillna(df['ilguc_log_ort']).fillna(genel)
    df = df.merge(trafo_profili(train, origin), on='tanim', how='left')
    df = df.merge(trafo_gun_profili(train, origin), on=['tanim', 'haftanin_gunu'], how='left')
    return df


def origin_seti(bas, bit, origin):
    hedef = train[(train['tarih'] >= bas) & (train['tarih'] <= bit)]
    return veri_seti_kur(hedef, pd.Timestamp(origin))


OZELLIKLER = [
    'guc', 'ay', 'gun', 'haftanin_gunu', 'hafta_sonu', 'tatil',
    'il', 'bolge', 'ilce', 'guc_bucket_str', 'grup_log_ort',
    'trafo_ortalama', 'trafo_medyan', 'trafo_gun_sayisi',
    'log_ortalama', 'log_medyan', 'log_std', 'sifir_orani',
    'son30_log_ort', 'son90_log_ort', 'son30_sifir', 'son90_sifir',
    'log_trend', 'trafo_gun_orani',
    'sic_7gun',
]
print('özellik sayısı:', len(OZELLIKLER))

özellik sayısı: 25


## Origin'ler

Bir origin, "elimde şu tarihe kadar veri var, sonrasını tahmin et" demek. Aynı hedef
dönemi farklı origin'lerle tekrar üretip yığıyoruz; model hem daha çok örnek görüyor
hem de aynı takvim gününü farklı profil tazeliğinde görmüş oluyor.

İki gruptan oluşuyor:

- **Yaz (Nisan/Mayıs/Haziran 2025, hedef 30 Eylül'e kadar):** Test dönemi Nisan-Temmuz.
  Bu origin'ler olmadan model yaz seviyelerini hiç görmüyor — eklendiğinde skor
  1.13709'dan 1.08157'ye düşmüştü, en büyük tek kazanç buydu.
- **Kış (Aralık 2025 / Ocak / Şubat 2026, hedef 31 Mart'a kadar):** Bunlarda profiller
  zengin (11-13 aylık geçmiş), gerçek testteki duruma (15 ay) en yakın örnekler bunlar.

Nisan 2025'ten önceki origin'ler denendi ve **zarar verdi** — o tarihlerde profiller
sadece 1-3 aylık, model yanlış bir rejimi öğreniyor. Daha fazla veri her zaman daha
iyi değil; profil kalitesi veri miktarından önemli çıktı.

In [5]:
ORIGINLER = [
    ('2025-04-01', '2025-09-30'),
    ('2025-05-01', '2025-09-30'),
    ('2025-06-01', '2025-09-30'),
    ('2025-12-01', '2026-03-31'),
    ('2026-01-01', '2026-03-31'),
    ('2026-02-01', '2026-03-31'),
]

egitim = pd.concat([origin_seti(bas, bit, bas) for bas, bit in ORIGINLER], ignore_index=True)
egitim['hedef_log'] = np.log1p(egitim['tuketim_duzeltilmis'])
for col in KATEGORIK:
    egitim[col] = egitim[col].astype('category')

print('eğitim:', egitim.shape)
print('eğitim ayları:', sorted(egitim['ay'].unique()))
print('test ayları  :', sorted(test['ay'].unique()))

eğitim: (2096870, 36)
eğitim ayları: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(12)]
test ayları  : [np.int32(4), np.int32(5), np.int32(6), np.int32(7)]


num_leaves=15 kritik. 122 günlük ufukta model çok hızlı ezberliyor: basit ağaçlar
trafoya özel örüntüleri ezberleyemediği için uzak ufukta daha iyi genelliyor. Kaggle'da
63 -> 31 -> 15 adımlarının her biri skoru iyileştirdi. Doğrulamada 7-10 bandında dipleniyor,
yani bu yön yaklaşık tükenmiş durumda.

Üç farklı tohumla eğitip log uzayında ortalama alıyoruz. Metrik log ölçekte
hesaplandığı için ortalamayı da orada almak gerekiyor.

In [6]:
AYAR = dict(num_leaves=15, learning_rate=0.05, n_estimators=87)
TOHUMLAR = [42, 7, 2024]

test_seti = veri_seti_kur(test, pd.Timestamp('2026-04-01'))

# LightGBM kategorileri kod numarasıyla kullanıyor. Test'in kategori sıralaması eğitimden
# farklı olursa model sessizce yanlış tahmin üretir - hata vermez. Bu yüzden birebir eşitliyoruz.
for col in KATEGORIK:
    test_seti[col] = pd.Categorical(test_seti[col], categories=egitim[col].cat.categories)

print('kategori uyumu:', all(list(test_seti[c].cat.categories) == list(egitim[c].cat.categories)
                             for c in KATEGORIK))
print('test\'te eksik sıcaklık:', int(test_seti['sic_7gun'].isna().sum()))

kategori uyumu: True
test'te eksik sıcaklık: 0


In [7]:
log_tahminler = []
for tohum in TOHUMLAR:
    model = lgb.LGBMRegressor(random_state=tohum, verbose=-1, **AYAR)
    model.fit(egitim[OZELLIKLER], egitim['hedef_log'], categorical_feature=KATEGORIK)
    log_tahminler.append(model.predict(test_seti[OZELLIKLER]))

# Negatif tahminler yarışma kuralı gereği 0 sayılıyor
tahmin = np.clip(np.expm1(np.mean(log_tahminler, axis=0)), 0, None)
submission = pd.DataFrame({'id': test_seti['id'], 'tuketim': tahmin})

ornek = pd.read_csv(RAW + 'sample_submission.csv')
print('satır sayısı eşit:', len(submission) == len(ornek))
print('id kümeleri aynı :', set(submission['id']) == set(ornek['id']))
print('NaN / negatif    :', submission['tuketim'].isna().sum(), '/', (submission['tuketim'] < 0).sum())
print()
print(submission['tuketim'].describe().round(1).to_string())

submission.to_csv('../submissions/sub_22_tek_hava_sic7gun.csv', index=False)
print('\nkaydedildi')

satır sayısı eşit: True
id kümeleri aynı : True
NaN / negatif    : 0 / 0

count    714688.0
mean       1727.8
std        2658.0
min           0.1
25%         433.8
50%        1138.9
75%        2332.6
max       72034.6

kaydedildi


In [8]:
onem = pd.DataFrame({'ozellik': OZELLIKLER, 'onem': model.feature_importances_})
print(onem.sort_values('onem', ascending=False).to_string(index=False))

         ozellik  onem
   son30_log_ort   279
        sic_7gun   132
            ilce   126
           bolge    77
  trafo_ortalama    76
             guc    73
    trafo_medyan    59
trafo_gun_sayisi    55
       log_trend    52
              ay    45
    log_ortalama    38
    grup_log_ort    37
   son90_log_ort    33
 trafo_gun_orani    31
         log_std    31
     son30_sifir    24
  guc_bucket_str    17
     son90_sifir    15
     sifir_orani    15
      log_medyan     3
             gun     0
           tatil     0
   haftanin_gunu     0
      hafta_sonu     0
              il     0
